# 辞書の欠損キーの処理には in や KeyError ではなく get を使う

辞書を扱うための 3つの基本的な演算とは、キーとその値へのアクセス、代入、削除です。辞書の内容は動的に変化し、キーにアクセスしたり、削除したりしようとしたのに、そのキーが既に存在しないことがあるし、それが普通です。

例えば、サンドイッチ屋のメニューを作るために、人々の好きなパンの種類を調べようとしているとしましょう。次のコードでは、種類ごとに現在の投票数を示す counters という辞書を定義します。

In [1]:
counters = {
  'pumpernickel': 2,
  'sourdough': 1,
}

新たな投票に対応してカウンタを増やすには、キーがあるかどうかを調べ、なければキーをデフォルトのカウンタ値 0 で挿入し、カウンタの値を 1つ増やします。これには、キーに2回アクセスして、1回代入する必要があります。次のコードでは、if 文でキーが存在する場合に True を返す in 式を使います。

In [2]:
key = 'wheat'

if key in counters:
  count = counters[key]
else:
  count = 0

counters[key] = count + 1

同じことを実現する別の方法に、存在しないキーの値を得たい場合に KeyError 例外を送出するという辞書の振る舞いを使うものがあります。この方式は 1回だけアクセスして 1回代入するので、より効率的です。

In [4]:
try:
  count = counters[key]
except KeyError:
  count = 0

counters[key] = count + 1

存在するキーを取得するかデフォルト値を返すというこの処理の流れはとても一般的なので、組み込み型の dict にはこの作業を行う get メソッドが用意されています。
これも 1回のアクセスと 1回の代入だけですが、KeyError の場合よりはるかに短くて済みます。

In [6]:
count = counters.get(key, 0) # key が存在しない場合、デフォルト値 0 を返す
counters[key] = count + 1

in 式や KeyError 方式を短くすることは、さまざまな方法で可能ですが、どのような方式でも代入の重複という問題から逃れられず、読みにくいので避けるのが賢明です。

よって、単純な型の辞書では、get メソッドを使うのが最短で最も明確なコードになります。

辞書の値が list のようなもっと複雑な型ならどうでしょうか。例えば、投票数を数えるだけでなく、だれがそのパンに投票したかも知りたいとします。
次のコードでは、各キーに名前の list を関連付けて、それを行います。

In [11]:
votes = {
  'baguette': ['Alice', 'Bob'],
  'ciabatta': ['Coco', 'Deb'],
}

key = 'brioche'
who = 'Eimer'

if key in votes:
  names = votes[key]
else:
  # キーがない場合は、空リストを代入
  votes[key] = names = []

names.append(who)
print(votes)

{'baguette': ['Alice', 'Bob'], 'ciabatta': ['Coco', 'Deb'], 'brioche': ['Eimer']}


get を使って list 値を取得するこの方式は、if 文で代入式を使えば、1行短縮することができ、読みやすさが向上します。

In [10]:
votes = {
  'baguette': ['Alice', 'Bob'],
  'ciabatta': ['Coco', 'Deb'],
}

if (names := votes.get(key)) is None:
  votes[key] = names = []

names.append(who)
print(votes)

{'baguette': ['Alice', 'Bob'], 'ciabatta': ['Coco', 'Deb'], 'brioche': ['Eimer']}


dict 型には、このパターンの処理をさらに簡潔に書ける setdefault メソッドもあります。
setdefault は、辞書のキー値を取得しようとします。そのキーがないと、このメソッドはそのキーに対して指定されたデフォルト値を割り当て、メソッドはそのキーに対する値を返します。

In [13]:
votes = {
  'baguette': ['Alice', 'Bob'],
  'ciabatta': ['Coco', 'Deb'],
}

names = votes.setdefault(key, [])
names.append(who)
print(votes)

{'baguette': ['Alice', 'Bob'], 'ciabatta': ['Coco', 'Deb'], 'brioche': ['Eimer']}


この方式は読みやすくありません。メソッド名 setdefault が目的を混乱させます。値を取得しているのに、どうして set なのでしょうか。
Python に詳しくない人には、setdefault が自明でないため、何をしようとしているのかわからないのではないかと心配になることです。

もう1つ重要なことがあります。setdefault に渡されるデフォルト値は欠損キーの場合には辞書に複製されるのではなく、直接代入されます。値が list の場合のその影響を次に示します。

In [14]:
data = {}
key = 'foo'
value = []
data.setdefault(key, value)
print('Before:', data)
value.append('hello')
print('After:', data)

Before: {'foo': []}
After: {'foo': ['hello']}


これは、setdefault でアクセスする場合は、どんなキーであっても新たなデフォルト値を作成しておかなければならない、ということです。

誰が投票したのかリストの代わりに辞書の値にカウンタを使っていた前の例に戻ると、この場合に setdefault を使ったらどうでしょうか。次に、そうしたコードの実装を示します。

In [15]:
count = counters.setdefault(key, 0)
counters[key] = count + 1

このコードにおける問題は、setdefault の呼び出しが余計なことだということです。カウンタを増やした後で、辞書のキーに常に新たな値を代入する必要があります。
カウンタ更新に get を使う従来の方式では、カウンタ更新は 1つのアクセスと 1つの代入だけでした。setdefault を使うと、1つのアクセスと 2つの代入が必要です。

## get と setdefault の違い

In [16]:
# get の場合（辞書は一回だけ更新）
counters[key] = counters.get(key, 0) + 1

- get は 辞書を書き換えない
- 上書きは最後の 1 回だけ → 無駄が少ない

In [17]:
# setdefault の場合（辞書が2回更新されうる）
counters[key] = counters.setdefault(key, 0) + 1

- キーがない場合：setdefault が 0 をセットして辞書を更新
- その後、counters[key] = ... で もう一度更新

| 状況   | get  | setdefault   |
| ---- | ---- | ------------ |
| キーあり | 更新1回 | 更新1回（実質同じ）   |
| キーなし | 更新1回 | **更新2回（無駄）** |


## 覚えておくこと

- 辞書の欠損キーを検出するには次の 4つの方法がある。in 式、KeyError例外、getメソッド、および setdefaultメソッドだ。
- get メソッドは、カウンタのような基本的な方からなる辞書に最適だ。辞書の値の生成コストがかかる場合や例外が送出される可能性がある場合は代入式と一緒に使うのが好ましい。
- 問題を解決するために dict の setdefault が最良と思われる場合は、代わりに defaultdict を使うことを検討する。